In [1]:
import os
from pathlib import Path
import pandas as pd

# Project root (auto-detect)
BASE_PATH = Path.cwd().parents[0]

# Data paths
RAW_PATH = BASE_PATH / "data" / "raw"
BRONZE_PATH = BASE_PATH / "data" / "bronze"

# Ensure bronze directory exists
BRONZE_PATH.mkdir(parents=True, exist_ok=True)

In [2]:
files = list(BRONZE_PATH.glob("final_combined_df_*.csv"))

final_combined_df = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

print("Final LA dataset shape:", final_combined_df.shape)

Final LA dataset shape: (4177218, 18)


In [3]:
files

[WindowsPath('E:/C-DAC/Major Project/AI-Based-Maritime-Port-Intelligence/data/bronze/final_la_df_aashi.csv'),
 WindowsPath('E:/C-DAC/Major Project/AI-Based-Maritime-Port-Intelligence/data/bronze/final_la_df_Harshit.csv'),
 WindowsPath('E:/C-DAC/Major Project/AI-Based-Maritime-Port-Intelligence/data/bronze/final_la_df_mayur.csv'),
 WindowsPath('E:/C-DAC/Major Project/AI-Based-Maritime-Port-Intelligence/data/bronze/final_la_df_Prapti.csv'),
 WindowsPath('E:/C-DAC/Major Project/AI-Based-Maritime-Port-Intelligence/data/bronze/final_la_df_yash.csv')]

In [4]:
final_combined_df

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,location_type
0,368237190,2024-03-01T00:00:10,33.74606,-118.22473,0.0,101.0,191.0,GO BEYOND,IMO9622655,WDM7781,70.0,0.0,55.0,15.0,3.3,70.0,A,port
1,477698600,2024-03-01T00:00:26,33.75748,-118.21417,0.0,136.9,0.0,COSCO DENMARK,IMO9516478,VRNP8,71.0,5.0,366.0,52.0,13.4,71.0,A,port
2,367719640,2024-03-01T00:00:01,33.74927,-118.27366,0.0,320.5,320.0,CATALINA PROVIDER,IMO8793562,WDI6530,70.0,9.0,45.0,15.0,1.7,70.0,A,port
3,413349820,2024-03-01T00:00:39,33.74363,-118.20785,0.0,105.3,309.0,QIAN KUN,IMO9432165,BQDS,70.0,5.0,200.0,28.0,10.2,70.0,A,port
4,373928000,2024-03-01T00:00:59,33.73807,-118.21567,0.0,303.0,249.0,GENIUS STAR XI,IMO9622710,3FGS3,71.0,1.0,124.0,22.0,6.0,70.0,A,port
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4177213,355586000,2024-10-31T22:56:18,33.74369,-118.20807,0.1,131.2,308.0,DAIWAN WISDOM,IMO9427134,3FWQ8,70.0,5.0,175.0,29.0,6.6,70.0,A,anchorage
4177214,355586000,2024-10-31T23:02:19,33.74370,-118.20809,0.1,110.0,308.0,DAIWAN WISDOM,IMO9427134,3FWQ8,70.0,5.0,175.0,29.0,6.6,70.0,A,anchorage
4177215,355586000,2024-10-31T23:11:18,33.74369,-118.20807,0.1,137.0,308.0,DAIWAN WISDOM,IMO9427134,3FWQ8,70.0,5.0,175.0,29.0,6.6,70.0,A,anchorage
4177216,416482000,2024-10-31T23:21:17,33.72659,-118.25960,0.1,45.3,201.0,EVER LOGIC,IMO9604081,BKIF,74.0,5.0,335.0,46.0,11.1,74.0,A,anchorage


In [5]:
out_file = BRONZE_PATH / f"final_combined_df.csv"
final_combined_df.to_csv(out_file, index=False)

In [6]:
final_combined_df.columns

Index(['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'Heading',
       'VesselName', 'IMO', 'CallSign', 'VesselType', 'Status', 'Length',
       'Width', 'Draft', 'Cargo', 'TransceiverClass', 'location_type'],
      dtype='object')

In [7]:
final_combined_df["BaseDateTime"] = pd.to_datetime(final_combined_df["BaseDateTime"])

In [8]:
final_combined_df["location_type"].value_counts()

location_type
port         3744972
anchorage     432246
Name: count, dtype: int64

In [10]:
final_combined_df["SOG"] = pd.to_numeric(final_combined_df["SOG"], errors="coerce")


In [11]:
final_combined_df["Length"] = pd.to_numeric(final_combined_df["Length"], errors="coerce")
final_combined_df = final_combined_df.dropna(subset=["Length"])


In [13]:
final_combined_df = final_combined_df.dropna(subset=["IMO", "VesselName"])

In [14]:
final_combined_df.shape

(4133628, 18)

In [16]:
required_cols = [
    "MMSI",
    "BaseDateTime",
    "location_type",
    "SOG",
    "Length",
    "IMO",
    "VesselName"
]

missing = [c for c in required_cols if c not in final_combined_df.columns]
print("Missing columns:", missing)

print(final_combined_df.dtypes)


Missing columns: []
MMSI                         int64
BaseDateTime        datetime64[ns]
LAT                        float64
LON                        float64
SOG                        float64
COG                        float64
Heading                    float64
VesselName                  object
IMO                         object
CallSign                    object
VesselType                 float64
Status                     float64
Length                     float64
Width                      float64
Draft                      float64
Cargo                      float64
TransceiverClass            object
location_type               object
dtype: object


In [17]:
import uuid
import numpy as np
import pandas as pd

port_calls = []

for mmsi, vessel in final_combined_df.groupby("MMSI"):
    
    vessel = vessel.sort_values("BaseDateTime")

    port_entries = vessel[vessel["location_type"] == "port"]
    anchorage_entries = vessel[vessel["location_type"] == "anchorage"]

    if port_entries.empty:
        continue

    arrival_time = port_entries["BaseDateTime"].min()
    departure_time = port_entries["BaseDateTime"].max()

    berth_start = port_entries[port_entries["SOG"] == 0]["BaseDateTime"].min()

    anchorage_wait = 0
    if not anchorage_entries.empty:
        anchorage_wait = (
            anchorage_entries["BaseDateTime"].max()
            - anchorage_entries["BaseDateTime"].min()
        ).total_seconds() / 60

    length = vessel["Length"].iloc[0]
    handled_teu = int((length / 400) * np.random.randint(6000, 12000))

    port_calls.append({
        "PortCallID": str(uuid.uuid4()),
        "MMSI": mmsi,
        "IMO": vessel["IMO"].iloc[0],
        "VesselName": vessel["VesselName"].iloc[0],
        "PortCode": "USLAX",
        "ArrivalTime": arrival_time,
        "BerthStartTime": berth_start,
        "DepartureTime": departure_time,
        "AnchorageWaitingTime": round(anchorage_wait, 2),
        "BerthTime": round(
            (departure_time - berth_start).total_seconds() / 3600, 2
        ) if pd.notna(berth_start) else 0,
        "HandledTEU": handled_teu,
        "PortCallStatus": "Completed" if anchorage_wait < 1440 else "Delayed"
    })

port_call_df = pd.DataFrame(port_calls)


In [19]:
port_call_df.head()

,PortCallID,MMSI,IMO,VesselName,PortCode,ArrivalTime,BerthStartTime,DepartureTime,AnchorageWaitingTime,BerthTime,HandledTEU,PortCallStatus
0,e3e10dc8-bd0d-4662-a61a-dfbcb74651ff,205717000,IMO9748485,LA TONDA,USLAX,2024-04-21 13:32:10,2024-04-21 13:32:10,2024-05-01 06:14:59,21083.23,232.71,4041,Delayed
1,ea88ad6b-8c52-41ae-a23c-c5cf4b6f4686,209652000,IMO9975466,MORPHOU,USLAX,2024-11-18 05:24:42,2024-11-18 05:24:42,2024-11-20 07:57:44,3039.88,50.55,4323,Delayed
2,e363892f-003a-4294-954c-c5d985082148,209889000,IMO9317834,XENIA,USLAX,2024-08-09 00:04:45,2024-08-09 00:04:45,2024-08-11 08:02:29,3363.40,55.96,4099,Delayed
3,75e0ed44-1fe6-44fc-9327-5e5d4322053b,210136000,IMO9454917,KNOSSOS WAVE,USLAX,2024-09-16 14:24:08,2024-09-16 14:24:08,2024-09-22 09:00:04,8327.50,138.60,5249,Delayed
4,788979d2-6a5b-41d0-a1cb-cd65633aa0c3,210188000,IMO9542491,PARASKEVI 2,USLAX,2024-06-05 17:55:44,2024-06-05 17:55:44,2024-06-16 00:27:12,14806.63,246.52,4727,Delayed


In [21]:
port_call_df.shape

(1112, 12)

In [20]:
final_combined_df["MMSI"].nunique()

1276

In [25]:
# Save Port Call data to bronze layer
port_call_df.to_csv(
    BRONZE_PATH / "la_port_calls_2024.csv",
    index=False
)


In [29]:
import numpy as np
import pandas as pd

# -------------------------------------------------
# SAFELY prepare Date column (NO warning)
# -------------------------------------------------
final_combined_df = final_combined_df.copy()

final_combined_df["Date"] = pd.to_datetime(
    final_combined_df["BaseDateTime"]
).dt.date

# -------------------------------------------------
# FULL YEAR coverage (2024 – leap year)
# -------------------------------------------------
unique_dates = sorted(final_combined_df["Date"].unique())

print("Number of days:", len(unique_dates))      # Expected: 366
print("Sample dates:", unique_dates[:5])

# -------------------------------------------------
# Generate FULL-YEAR Port Operations data
# -------------------------------------------------
port_ops = []

for date in unique_dates:
    for terminal in range(1, 8):        # 7 terminals
        for berth in range(1, 11):      # 10 berths per terminal

            berth_length = np.random.randint(300, 420)
            berth_depth = round(np.random.uniform(14, 18), 1)

            port_ops.append({
                "PortCode": "USLAX",
                "TerminalID": f"T{terminal}",
                "BerthID": f"B{berth}",
                "BerthLength": berth_length,
                "BerthDepth": berth_depth,
                "MaxLOA": berth_length - 10,
                "BerthMaxDraft": berth_depth - 1,
                "DailyCapacityTEU": np.random.randint(8000, 20000),
                "OperationalStatus": np.random.choice(
                    ["Normal", "Congested", "Maintenance"],
                    p=[0.65, 0.30, 0.05]
                ),
                "Date": date
            })

# -------------------------------------------------
# Create Port Operations DataFrame
# -------------------------------------------------
port_ops_df = pd.DataFrame(port_ops)

# -------------------------------------------------
# Sanity checks (IMPORTANT)
# -------------------------------------------------
print("Port Ops shape:", port_ops_df.shape)
print("Unique dates:", port_ops_df["Date"].nunique())
print("Terminals:", port_ops_df["TerminalID"].nunique())
print("Berths per terminal:", port_ops_df["BerthID"].nunique())


Number of days: 366
Sample dates: [datetime.date(2024, 1, 1), datetime.date(2024, 1, 2), datetime.date(2024, 1, 3), datetime.date(2024, 1, 4), datetime.date(2024, 1, 5)]
Port Ops shape: (25620, 10)
Unique dates: 366
Terminals: 7
Berths per terminal: 10


In [30]:
# Create DataFrame
port_ops_df = pd.DataFrame(port_ops)

# Sanity check
print("Port Ops shape:", port_ops_df.shape)
print("Unique dates in Port Ops:", port_ops_df["Date"].nunique())

Port Ops shape: (25620, 10)
Unique dates in Port Ops: 366


In [32]:
port_ops_df.to_csv(
    BRONZE_PATH / "la_port_oprs_2024.csv",
    index=False
)
